# Multi step workflow with Language Chain Expression Language

Create an AI Business Advisor that:

* Accepts an industry as input.
* Generates a business idea.
* Analyzes the strengths and weaknesses.
* Formats the results as a final report.

Use LCEL to chain prompts LLMs and output parsers.

In [ ]:
from dotenv import load_dotenv

loaded = load_dotenv("../.env")
print("Loaded:", loaded)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field
import os

In [ ]:
# intantiate the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openai.vocareum.com/v1"
)

## Create a chain
This runnable should parse the provided input and also log it. So parsing and logging can run in parallel. Hence use RunnableParallel.

RunnableParallel, expects a map of steps. Below we have output and log but these names can be also different.

In [ ]:
logs = []

In [ ]:
parser = StrOutputParser()

In [ ]:
parse_and_log_output_chain = RunnableParallel(
    output=parser,
    log=RunnableLambda(lambda x: logs.append(x))
)

## Idea Generation

Given industry LLM should generate some idea.

In [ ]:
idea_prompt = PromptTemplate(
    template=("""You are an experienced business consultant.
    
Generate one innovative and practical business idea in the {industry} industry.

Provide a brief description of the idea.""")
)

In [ ]:
idea_chain = idea_prompt | llm | parse_and_log_output_chain

In [ ]:
idea_result = idea_chain.invoke("agro")

In [ ]:
idea_result["output"]

In [ ]:
logs

## Idea analysis
Now llm should analyze the generated output from idea generation step.


In [ ]:
analysis_prompt = PromptTemplate(
    template=(
        """Please analyze the following idea: {idea}
        And create only 3 strengths and 3 weaknesses of the idea.
        """
    )
)

In [ ]:
analysis_chain = analysis_prompt | llm | parse_and_log_output_chain

In [ ]:
analysis_result = analysis_chain.invoke(idea_result["output"])

In [ ]:
analysis_result["output"]

In [ ]:
for message in logs:
    print(message.content)
    print("-----")

## Report

Create a business report from the strengts and weaknesses of the business idea.

In [ ]:
report_prompt = PromptTemplate(
    template=(
        """Here is the business analysis:
        Strengths&Weaknesses: {output}
        
        Generate structured business report""" 
    )
)

In [ ]:
class AnalysisReport(BaseModel):
    """Strengths and weaknesses about a business idea."""
    strengths: list = Field(default=[], description="Idea's strengths list")
    weaknesses: list = Field(default=[], description="Idea's weaknesses list")

In [ ]:
report_chain = (report_prompt | llm.with_structured_output(AnalysisReport, method="function_calling"))

In [ ]:
report_result = report_chain.invoke(analysis_result["output"])

In [ ]:
report_result

In [ ]:
report_result.strengths

In [ ]:
report_result.weaknesses

## E2E Chain

In [ ]:
from langchain_core.runnables import RunnableLambda

e2e_chain = (
    idea_chain
    | RunnableLambda(lambda x: {"idea": x})
    | analysis_chain
    | report_chain
)

In [ ]:
report = e2e_chain.invoke("interior design")

In [ ]:
report

In [ ]:
report.strengths

In [ ]:
report.weaknesses

In [ ]:
e2e_chain.get_graph().print_ascii()